# Global Code

In [ ]:
from pydantic import BaseModel, SkipValidation
from functools import cache
from transformers import AutoTokenizer, AutoModelForCausalLM

# Refrain from calling these inside the notebook if cache is not available. 
# Takes too long. Use the console instead
from contextlib import redirect_stdout
import io

class ModelConfig(BaseModel):
    # This is required to set the member typehints to Auto*
    model_config = dict(arbitrary_types_allowed=True)

    model_id: str
    tokenizer: SkipValidation[AutoTokenizer]
    model: SkipValidation[AutoModelForCausalLM]
    
    # The order of decorators does matter
    @classmethod
    @cache
    def from_pretrained(cls, model_id: str):
        with redirect_stdout(io.StringIO()):
            return ModelConfig(            
                model_id=model_id,
                tokenizer=AutoTokenizer.from_pretrained(
                    pretrained_model_name_or_path=model_id, 
                    # NEVER forget this for causal models
                    padding_side="left"
                ),
                model=AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id)
            )
    
    def generate(self, query: str, max_new_tokens=512, temperature=.7, top_p=.9) -> str:
        inputs = self.tokenizer(query, return_tensors="pt")
        token_ids = self.model.generate(
            **inputs, 
            # Check https://huggingface.co/docs/transformers/llm_tutorial
            do_sample=True, 
            max_new_tokens=max_new_tokens, 
            temperature=temperature, 
            top_p=top_p,
            num_beams=4
        )
        return self.tokenizer.batch_decode(
            token_ids, 
            skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

# `meta-llama/Llama-3.2-1B-Instruct`

In [61]:
llm = ModelConfig.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

In [62]:
# Call generate using a plain and an annotated prompt
plain_prompt = "Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence."

annotated_prompt = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [ ]:
plain_response = llm.generate(query=plain_prompt)
annotated_response = llm.generate(query=annotated_prompt)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


# `meta-llama/Llama-3.2-1B`

In [69]:
llm = ModelConfig.from_pretrained("meta-llama/Llama-3.2-1B")

In [70]:
# Call generate using a plain and an annotated prompt
plain_prompt = "Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence."

annotated_prompt = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Write 5 sentences of at least 5 words each, so that each sentence contains at least 1 word more than its orevious sentence.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [71]:
plain_response = llm.generate(query=plain_prompt)
annotated_response = llm.generate(query=annotated_prompt)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
import os
print(annotated_response[0][:128], plain_response[0][:128], sep=os.linesep)

# ARCV

In [10]:
import os
import util

In [5]:
model = util.ModelConfig.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

In [16]:
# transformers.tokenization_utils_fast.PreTrainedTokenizerFast
type(model.tokenizer)

tokens = model.tokenizer("What is my name computer? I forgot it")
# transformers.tokenization_utils_base.BatchEncoding
type(tokens)

token_tensor = model.tokenizer("What is my name computer? I forgot it", return_tensors="pt")
# transformers.tokenization_utils_base.BatchEncoding
type(token_tensor)

print(tokens, token_tensor, sep=os.linesep)

{'input_ids': [128000, 3923, 374, 856, 836, 6500, 30, 358, 29695, 433], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
{'input_ids': tensor([[128000,   3923,    374,    856,    836,   6500,     30,    358,  29695,
            433]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [22]:
ret_tokens = model.model.generate(**token_tensor, max_new_tokens=512, temperature=.7, top_p=.9)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [24]:
from transformers import PreTrainedTokenizerFast

model.tokenizer.batch_decode(            
    ret_tokens,
    skip_special_tokens=True, 
    clean_up_tokenization_spaces=False
)

["What is my name computer? I forgot it.\nIf you're a human, you can try typing your name into the search bar on the computer to find your name.\n\nIf you're a computer, you're probably a custom-built machine with a specific model number and a unique identifier, such as a MAC address. In that case, you can try typing your model number and MAC address into the search bar to find your specific computer.\n\nIf you're a software or application, you can try searching for your name or your product in the search bar to find information about you.\n\nBut if you're a mysterious entity, like a ghost or a spirit, I'm not sure what to do!"]